# **This section will contain the code operations on the raw dataset.**

In [1]:
from pathlib import Path
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
        PROJECT_ROOT = candidate; break
else: raise RuntimeError("Start Jupyter inside this project so the repository root can be found.")
DATA_DIR=PROJECT_ROOT/"data"; RAW_DATA_DIR=DATA_DIR/"01_raw"; INTERMEDIATE_DIR=DATA_DIR/"02_intermediate"
MODEL_DIR=PROJECT_ROOT/"models"; REPORT_DIR=PROJECT_ROOT/"reports"
INTERMEDIATE_DIR.mkdir(parents=True,exist_ok=True); MODEL_DIR.mkdir(parents=True,exist_ok=True); REPORT_DIR.mkdir(parents=True,exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\acer\Desktop\Programs\Projects_for _data_science\Fraud_Detection_Model


In [2]:
# Importing all the required libraries:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline 
print("All required libraries are imported successfully!")

All required libraries are imported successfully!


### Why transaction and identity tables are loaded separately

The benchmark stores transaction facts and identity/device observations in separate tables. The transaction table is the event spine: each row is a payment event and contains the target in training. The identity table enriches only some events. Keeping these sources separate during inspection lets us see what each source contributes before joining them.

In [3]:
# Loading the dataset:
train_transaction = pd.read_csv(str(RAW_DATA_DIR / "train_transaction.csv"))

In [4]:
# Overview of the dataset:
train_transaction.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
#Summary of the dataset:
train_transaction.info()

<class 'pandas.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Columns: 394 entries, TransactionID to V339
dtypes: float64(376), int64(4), str(14)
memory usage: 1.7 GB


In [6]:
# Shape of the dataset:
train_transaction.shape

(590540, 394)

In [7]:
# Data Types of the dataset:
train_transaction.dtypes

TransactionID       int64
isFraud             int64
TransactionDT       int64
TransactionAmt    float64
ProductCD             str
                   ...   
V335              float64
V336              float64
V337              float64
V338              float64
V339              float64
Length: 394, dtype: object

In [8]:
# Statistical summary of the dataset:
train_transaction.describe()

,TransactionID,isFraud,TransactionDT,TransactionAmt,card1,card2,card3,card5,addr1,addr2,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
count,5.905400e+05,590540.000000,5.905400e+05,590540.000000,590540.000000,581607.000000,588975.000000,586281.000000,524834.000000,524834.000000,...,82351.000000,82351.000000,82351.000000,82351.000000,82351.000000,82351.00000,82351.000000,82351.000000,82351.000000,82351.000000
mean,3.282270e+06,0.034990,7.372311e+06,135.027176,9898.734658,362.555488,153.194925,199.278897,290.733794,86.800630,...,0.775874,721.741883,1375.783644,1014.622782,9.807015,59.16455,28.530903,55.352422,151.160542,100.700882
std,1.704744e+05,0.183755,4.617224e+06,239.162522,4901.170153,157.793246,11.336444,41.244453,101.741072,2.690623,...,4.727971,6217.223583,11169.275702,7955.735482,243.861391,387.62948,274.576920,668.486833,1095.034387,814.946722
min,2.987000e+06,0.000000,8.640000e+04,0.251000,1000.000000,100.000000,100.000000,100.000000,100.000000,10.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000
25%,3.134635e+06,0.000000,3.027058e+06,43.321000,6019.000000,214.000000,150.000000,166.000000,204.000000,87.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000
50%,3.282270e+06,0.000000,7.306528e+06,68.769000,9678.000000,361.000000,150.000000,226.000000,299.000000,87.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000
75%,3.429904e+06,0.000000,1.124662e+07,125.000000,14184.000000,512.000000,150.000000,226.000000,330.000000,87.000000,...,0.000000,0.000000,25.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000
max,3.577539e+06,1.000000,1.581113e+07,31937.391000,18396.000000,600.000000,231.000000,237.000000,540.000000,102.000000,...,55.000000,160000.000000,160000.000000,160000.000000,55125.000000,55125.00000,55125.000000,104060.000000,104060.000000,104060.000000


In [9]:
# columns of the dataset:
train_transaction.columns.tolist()

['TransactionID',
 'isFraud',
 'TransactionDT',
 'TransactionAmt',
 'ProductCD',
 'card1',
 'card2',
 'card3',
 'card4',
 'card5',
 'card6',
 'addr1',
 'addr2',
 'dist1',
 'dist2',
 'P_emaildomain',
 'R_emaildomain',
 'C1',
 'C2',
 'C3',
 'C4',
 'C5',
 'C6',
 'C7',
 'C8',
 'C9',
 'C10',
 'C11',
 'C12',
 'C13',
 'C14',
 'D1',
 'D2',
 'D3',
 'D4',
 'D5',
 'D6',
 'D7',
 'D8',
 'D9',
 'D10',
 'D11',
 'D12',
 'D13',
 'D14',
 'D15',
 'M1',
 'M2',
 'M3',
 'M4',
 'M5',
 'M6',
 'M7',
 'M8',
 'M9',
 'V1',
 'V2',
 'V3',
 'V4',
 'V5',
 'V6',
 'V7',
 'V8',
 'V9',
 'V10',
 'V11',
 'V12',
 'V13',
 'V14',
 'V15',
 'V16',
 'V17',
 'V18',
 'V19',
 'V20',
 'V21',
 'V22',
 'V23',
 'V24',
 'V25',
 'V26',
 'V27',
 'V28',
 'V29',
 'V30',
 'V31',
 'V32',
 'V33',
 'V34',
 'V35',
 'V36',
 'V37',
 'V38',
 'V39',
 'V40',
 'V41',
 'V42',
 'V43',
 'V44',
 'V45',
 'V46',
 'V47',
 'V48',
 'V49',
 'V50',
 'V51',
 'V52',
 'V53',
 'V54',
 'V55',
 'V56',
 'V57',
 'V58',
 'V59',
 'V60',
 'V61',
 'V62',
 'V63',
 'V64',
 'V

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": train_transaction.isna().sum(),
    "missing_pct": train_transaction.isna().mean().mul(100),
}).sort_values("missing_pct", ascending=False)
missing_summary.head(20)


In [11]:
for i in train_transaction.columns:
    print(f"Column: {i} | Missing Values: {train_transaction[i].isnull().sum() } | Percentage: {train_transaction[i].isnull().mean()*100:.2f}%")

Column: TransactionID | Missing Values: 0 | Percentage: 0.00%
Column: isFraud | Missing Values: 0 | Percentage: 0.00%
Column: TransactionDT | Missing Values: 0 | Percentage: 0.00%
Column: TransactionAmt | Missing Values: 0 | Percentage: 0.00%
Column: ProductCD | Missing Values: 0 | Percentage: 0.00%
Column: card1 | Missing Values: 0 | Percentage: 0.00%
Column: card2 | Missing Values: 8933 | Percentage: 1.51%
Column: card3 | Missing Values: 1565 | Percentage: 0.27%
Column: card4 | Missing Values: 1577 | Percentage: 0.27%
Column: card5 | Missing Values: 4259 | Percentage: 0.72%
Column: card6 | Missing Values: 1571 | Percentage: 0.27%
Column: addr1 | Missing Values: 65706 | Percentage: 11.13%
Column: addr2 | Missing Values: 65706 | Percentage: 11.13%
Column: dist1 | Missing Values: 352271 | Percentage: 59.65%
Column: dist2 | Missing Values: 552913 | Percentage: 93.63%
Column: P_emaildomain | Missing Values: 94456 | Percentage: 15.99%
Column: R_emaildomain | Missing Values: 453249 | Percen

In [12]:
#Column wise anlaysis:
for col in (train_transaction.columns):
    if train_transaction[col].isnull().sum() == 0:
        print(f"The nunique: {train_transaction[col].nunique()} | Column: {col} | Data Type: {train_transaction[col].dtype}")

        print(f'This is the column which contains absolute non-zero: {col}')
        

The nunique: 590540 | Column: TransactionID | Data Type: int64
This is the column which contains absolute non-zero: TransactionID
The nunique: 2 | Column: isFraud | Data Type: int64
This is the column which contains absolute non-zero: isFraud
The nunique: 573349 | Column: TransactionDT | Data Type: int64
This is the column which contains absolute non-zero: TransactionDT
The nunique: 20902 | Column: TransactionAmt | Data Type: float64
This is the column which contains absolute non-zero: TransactionAmt
The nunique: 5 | Column: ProductCD | Data Type: str
This is the column which contains absolute non-zero: ProductCD
The nunique: 13553 | Column: card1 | Data Type: int64
This is the column which contains absolute non-zero: card1
The nunique: 1657 | Column: C1 | Data Type: float64
This is the column which contains absolute non-zero: C1
The nunique: 1216 | Column: C2 | Data Type: float64
This is the column which contains absolute non-zero: C2
The nunique: 27 | Column: C3 | Data Type: float64


# **Test dataset column or target output for the dataset:**

In [13]:
train_identity = pd.read_csv(str(RAW_DATA_DIR / "train_identity.csv"))

In [14]:
train_identity.head()

,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987004,0.0,70787.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M
1,2987008,-5.0,98945.0,NaN,NaN,0.0,-5.0,NaN,NaN,NaN,...,mobile safari 11.0,32.0,1334x750,match_status:1,T,F,F,T,mobile,iOS Device
2,2987010,-5.0,191631.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,Windows
3,2987011,-5.0,221832.0,NaN,NaN,0.0,-6.0,NaN,NaN,NaN,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,NaN
4,2987016,0.0,7460.0,0.0,0.0,1.0,0.0,NaN,NaN,0.0,...,chrome 62.0,24.0,1280x800,match_status:2,T,F,T,T,desktop,MacOS


# Every concept goes in a way that, there is to be a combination of the dataset:
Consisting in a way: transaction + indentity 

### Why use a left join on `TransactionID`

The transaction is the unit we want to score, so the transaction table stays on the left. A left join keeps every payment even when identity data is unavailable; an inner join would silently discard those transactions and bias the sample toward events with device information. `TransactionID` is the join key, not a predictor: it identifies a row but should not help a model memorize it. Check row counts and key uniqueness after merging.

In [15]:
train = train_transaction.merge(train_identity,on='TransactionID',how='left')

In [16]:
train_identity.head()

,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987004,0.0,70787.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M
1,2987008,-5.0,98945.0,NaN,NaN,0.0,-5.0,NaN,NaN,NaN,...,mobile safari 11.0,32.0,1334x750,match_status:1,T,F,F,T,mobile,iOS Device
2,2987010,-5.0,191631.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,Windows
3,2987011,-5.0,221832.0,NaN,NaN,0.0,-6.0,NaN,NaN,NaN,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,NaN
4,2987016,0.0,7460.0,0.0,0.0,1.0,0.0,NaN,NaN,0.0,...,chrome 62.0,24.0,1280x800,match_status:2,T,F,T,T,desktop,MacOS


In [17]:
train

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.50,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.00,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.00,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.00,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.00,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,3577535,0,15811047,49.00,W,6550,NaN,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590536,3577536,0,15811049,39.50,W,10444,225.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590537,3577537,0,15811079,30.95,W,12037,595.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590538,3577538,0,15811088,117.00,W,7826,481.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# **TEST DATASET** : Combination of test Dataset : transaction and identity.

### Why mirror the same join for test

Training and test must have the same predictor structure so the later feature pipeline receives a consistent schema. The competition test file has no `isFraud` column by design. We do not fill in or infer those labels; we preserve the test rows for later scoring.

In [18]:
test_transaction = pd.read_csv(str(RAW_DATA_DIR / "test_transaction.csv"))

In [19]:
test_transaction.sample(10)

,TransactionID,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
3556,3667105,18544735,88.500,W,9500,321.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
291667,3955216,28501068,77.000,W,18132,567.0,150.0,mastercard,117.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
314054,3977603,29193287,7.123,C,2650,142.0,185.0,mastercard,224.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
479523,4143072,33621760,100.000,H,6128,111.0,150.0,mastercard,224.0,debit,...,1.0,100.0,100.0,100.0,0.0,0.0,0.0,100.0,100.0,100.0
67991,3731540,20721764,30.000,W,15399,390.0,150.0,mastercard,224.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
266347,3929896,27644277,44.500,W,9112,250.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
272287,3935836,27824890,97.000,W,11839,490.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
166322,3829871,24158425,554.000,W,10155,225.0,150.0,mastercard,224.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
441753,4105302,32919210,100.000,H,16659,170.0,150.0,visa,226.0,credit,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
168856,3832405,24244613,39.000,W,7508,321.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
test_identity = pd.read_csv(str(RAW_DATA_DIR / "test_identity.csv"))   

In [21]:
test_identity.sample(10)

,TransactionID,id-01,id-02,id-03,id-04,id-05,id-06,id-07,id-08,id-09,...,id-31,id-32,id-33,id-34,id-35,id-36,id-37,id-38,DeviceType,DeviceInfo
69957,3998205,-5.0,23219.0,NaN,NaN,22.0,-12.0,NaN,NaN,NaN,...,mobile safari 12.0,32.0,2208x1242,match_status:2,T,F,F,T,mobile,iOS Device
58295,3951666,-45.0,638930.0,0.0,0.0,0.0,-27.0,NaN,NaN,0.0,...,chrome 69.0 for android,NaN,NaN,NaN,F,F,T,F,mobile,SM-J730GM Build/NRD90M
110672,4113895,-20.0,501626.0,NaN,NaN,-10.0,-35.0,NaN,NaN,NaN,...,chrome 70.0 for ios,NaN,NaN,NaN,F,F,F,F,mobile,NaN
5139,3695971,-10.0,252108.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,chrome 67.0,NaN,NaN,NaN,F,F,T,F,desktop,Windows
76393,4021408,-5.0,53790.0,0.0,0.0,1.0,-5.0,NaN,NaN,0.0,...,chrome 70.0,NaN,NaN,match_status:2,F,F,T,T,desktop,Windows
43550,3894760,-50.0,103109.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,chrome 67.0 for android,NaN,NaN,NaN,F,F,T,F,mobile,BLN-L21 Build/HONORBLN-L21
132892,4150886,-5.0,184111.0,NaN,NaN,0.0,0.0,NaN,NaN,NaN,...,mobile safari 11.0,32.0,2048x1536,match_status:2,T,F,F,F,mobile,iOS Device
96258,4081503,-5.0,287579.0,0.0,0.0,2.0,-1.0,NaN,NaN,0.0,...,chrome 71.0 for android,24.0,1920x1080,match_status:2,T,F,T,F,mobile,SM-G930V
136655,4156791,-5.0,23866.0,NaN,NaN,0.0,0.0,NaN,NaN,NaN,...,firefox 64.0,NaN,NaN,NaN,F,F,T,F,desktop,Windows
79079,4030890,-45.0,380186.0,NaN,NaN,-6.0,-25.0,NaN,NaN,NaN,...,chrome 70.0 for android,NaN,NaN,NaN,F,F,T,F,mobile,MYA-L13


In [22]:
test = test_transaction.merge(test_identity,on='TransactionID',how='left')

In [23]:
test.sample(10)

,TransactionID,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,...,id-31,id-32,id-33,id-34,id-35,id-36,id-37,id-38,DeviceType,DeviceInfo
18114,3681663,18993437,59.00,W,18018,452.0,150.0,mastercard,117.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
302375,3965924,28866739,59.00,W,17188,321.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
433815,4097364,32736464,100.00,R,1333,170.0,150.0,visa,195.0,credit,...,chrome 71.0,24.0,1920x1080,match_status:2,T,F,T,F,desktop,Windows
402375,4065924,31936144,20.00,S,2722,NaN,150.0,visa,226.0,debit,...,chrome 70.0,24.0,1680x1050,match_status:2,T,F,T,T,desktop,MacOS
453453,4117002,33171228,49.00,W,17868,148.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
423948,4087497,32486658,117.00,W,3576,321.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
94747,3758296,21588582,21.95,W,16255,470.0,150.0,mastercard,137.0,credit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
37043,3700592,19667737,44.00,W,2803,100.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
85790,3749339,21310661,171.00,W,18132,567.0,150.0,mastercard,117.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
61323,3724872,20479015,54.50,W,7919,194.0,150.0,mastercard,166.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Transforming into intermediate as a specific data type: Using pickling

In [24]:
import pickle

### Why save an intermediate pickle

CSV is portable and inspectable but expensive to parse repeatedly. A pickle speeds up local notebook iteration and preserves pandas dtypes. It is a derived local artifact, not a source of truth: regenerate it from the raw CSVs if ingestion logic changes, and never load pickles from an untrusted source.

In [25]:
#training data is transformed into a pickle file for further processing and analysis.


train.to_pickle(str(INTERMEDIATE_DIR / "train.pkl"))

In [26]:
# testing data is transformed into a pickle file for further processing and analysis.

test.to_pickle(str(INTERMEDIATE_DIR / "test.pkl"))

# Finally, we have transformed our dataset into a whole new format consisting  all the required features and its target, collectively.

# Hence, our part of data ingestion is done till now after the integration of our whole dataset which is now ready to be processed further in its respective operational field.